In [ ]:
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from pathlib import Path

In [ ]:
file = 'Shared Expense App Design Evaluation Survey.xlsx'
df = pd.read_excel(file, sheet_name='Sheet1')
print('Participants:', len(df))

Participants: 20


In [ ]:
metrics = ['Usefulness', 'Ease of Understanding', 'Ease of Use', 'Likelihood of Use']
alternatives = ['Alternative 1', 'Alternative 2', 'Alternative 3']
useful = ['Not useful at all', 'Slightly useful', 'Moderately useful', 'Very useful', 'Extremely useful']
easy = ['Very difficult', 'Somewhat difficult', 'Neither easy nor difficult', 'Somewhat easy', 'Very easy']
likely = ['Very unlikely', 'Unlikely', 'Neither likely nor unlikely', 'Likely', 'Very likely']
scales = [useful, easy, easy, likely]
ratings = {}
for i, metric in enumerate(metrics):
    raw = df.iloc[:, [9+i, 17+i, 25+i]].copy()
    raw.columns = alternatives
    scores = raw.apply(lambda col: col.str.strip().map(dict(zip(scales[i], range(1,6)))))
    if scores.isna().any().any():
        raise ValueError(f'Missing or unrecognized rating: {metric}')
    ratings[metric] = scores

In [ ]:
means = pd.DataFrame({m: r.mean() for m,r in ratings.items()}).T
sd = pd.DataFrame({m: r.std() for m,r in ratings.items()}).T
print('\nMean ratings:\n', means.round(2))
print('\nStandard deviations:\n', sd.round(2))
preferences = pd.DataFrame({label: df.iloc[:,col].str.extract(r'Design (\d)', expand=False).value_counts().reindex(['1','2','3'], fill_value=0).values for label,col in [('Most Useful',31),('Easiest to Use',33)]}, index=alternatives).T
percentages = preferences.div(preferences.sum(axis=1),axis=0)*100
print('\nPreferences (%):\n', percentages)


Mean ratings:
                        Alternative 1  Alternative 2  Alternative 3
Usefulness                      4.00           3.25           3.70
Ease of Understanding           4.35           3.80           4.30
Ease of Use                     4.25           3.50           4.50
Likelihood of Use               4.00           3.05           3.85

Standard deviations:
                        Alternative 1  Alternative 2  Alternative 3
Usefulness                      0.86           1.07           0.86
Ease of Understanding           0.81           1.32           0.86
Ease of Use                     0.79           1.19           0.61
Likelihood of Use               0.79           1.10           0.99

Preferences (%):
                 Alternative 1  Alternative 2  Alternative 3
Most Useful              70.0            5.0           25.0
Easiest to Use           35.0           15.0           50.0


In [ ]:
results = []
for metric, r in ratings.items():
    stat,p = friedmanchisquare(*(r[a] for a in alternatives))
    results.append([metric,stat,2,p])
results = pd.DataFrame(results, columns=['Metric','Chi-square','df','p'])
results['Adjusted p'] = multipletests(results['p'],method='holm')[1]
print('\nFriedman tests:\n',results.to_string(index=False))


for metric in results.loc[results['Adjusted p'] < .05,'Metric']:
    r = ratings[metric]
    rows = []
    for a,b in combinations(alternatives,2):
        stat,p = wilcoxon(r[a],r[b],alternative='two-sided',zero_method='wilcox',method='approx')
        rows.append([a+' vs '+b,stat,p])
    post = pd.DataFrame(rows,columns=['Comparison','W','p'])
    post['Adjusted p'] = multipletests(post['p'],method='holm')[1]
    print('\n'+metric+'\n',post.to_string(index=False))


Friedman tests:
                Metric  Chi-square  df        p  Adjusted p
           Usefulness    8.779661   2 0.012403    0.024806
Ease of Understanding    3.500000   2 0.173774    0.173774
          Ease of Use   12.688525   2 0.001757    0.007027
    Likelihood of Use   11.838710   2 0.002687    0.008061

Usefulness
                     Comparison    W        p  Adjusted p
Alternative 1 vs Alternative 2 15.5 0.017097    0.051291
Alternative 1 vs Alternative 3 28.5 0.209647    0.209647
Alternative 2 vs Alternative 3 33.0 0.102941    0.205883

Ease of Use
                     Comparison    W        p  Adjusted p
Alternative 1 vs Alternative 2 16.5 0.018258    0.036517
Alternative 1 vs Alternative 3 18.0 0.131668    0.131668
Alternative 2 vs Alternative 3 12.0 0.001570    0.004710

Likelihood of Use
                     Comparison    W        p  Adjusted p
Alternative 1 vs Alternative 2 14.0 0.001790    0.005370
Alternative 1 vs Alternative 3 30.0 0.466854    0.466854
Alternative 2